# 01 — Exploration de CUAD v1

**Objectif** : comprendre la structure et le contenu de CUAD **avant** tout entraînement de modèle,
en particulier pour nos trois familles de clauses (durée, résiliation, responsabilité).

**Données** : CUAD v1 (The Atticus Project, CC BY 4.0), téléchargé le 2026-11-16 dans
`data/raw/cuad/CUAD_v1/` (non versionné). Source, empreinte MD5, licence et limites :
voir [docs/data_provenance.md](../docs/data_provenance.md).

**Questions étudiées** (une section chacune) :

- **a.** Quelle est la structure de `CUAD_v1.json` ? (contrats, questions, réponses, exemple)
- **b.** Quelle est la longueur des contrats en caractères ?
- **c.** Pour chacune des 41 catégories, combien de contrats ont au moins une annotation ?
- **d.** À quoi ressemblent de vrais passages annotés pour nos familles ?
- **e.** Quelle est la longueur des passages annotés, par catégorie de nos familles ?
- **f.** Combien de contrats contiennent à la fois *Cap On Liability* et *Uncapped Liability* ?
- **g.** Comment expliquer l'écart entre nos 13 823 passages et les 13 101 annotations de l'article ?
- **h.** Le champ `context` du JSON est-il identique au fichier de `full_contract_txt/` ?

> **Exploration descriptive uniquement.** Ce notebook décrit les données ; il ne sert à régler
> **aucun** modèle (ni hyperparamètre, ni seuil, ni choix de modèle). Il porte sur l'ensemble de
> CUAD, sans découpage. Le découpage entraînement / validation / test sera décidé ensuite
> (décision **D-004**), et le jeu de test ne sera jamais utilisé pour régler un modèle (règle 2).

> **Avant tout commit de ce notebook** : ses sorties affichent des extraits de contrats.
> Les effacer d'abord (dans VS Code : **Clear All Outputs**), pour ne publier aucun texte de
> contrat dans le dépôt public (règle 5 et D-003).

**Comment l'exécuter** : une cellule à la fois, **dans l'ordre**, avec `Maj + Entrée`.
Le noyau doit être le `.venv` (Python 3.11).

## 0. Préparation

**Ce que fait la cellule** : elle importe les bibliothèques, indique où se trouvent les données,
et définit nos familles de clauses (la liste des catégories CUAD qui nous intéressent).

**Ce qu'on cherche à vérifier** : que le notebook trouve bien le dossier CUAD (`True`)
et qu'il utilise pandas 3 (celui du `.venv`).

In [ ]:
# Bibliothèques standard de Python (aucune installation nécessaire)
import ast                # lire des listes écrites sous forme de texte dans le CSV (question g)
import json               # lire le fichier CUAD_v1.json
from pathlib import Path  # manipuler des chemins de fichiers

# Bibliothèques installées dans le .venv (voir requirements.txt)
import matplotlib.pyplot as plt
import pandas as pd

# VS Code lance le notebook depuis le dossier notebooks/ :
# la racine du projet est alors le dossier parent.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CUAD_DIR = ROOT / 'data' / 'raw' / 'cuad' / 'CUAD_v1'

# Nos trois familles de clauses et les catégories CUAD correspondantes.
# La confidentialité n'a pas de catégorie dans CUAD (voir D-002).
FAMILLES = {
    'durée': ['Agreement Date', 'Effective Date', 'Expiration Date',
              'Renewal Term', 'Notice Period To Terminate Renewal'],
    'résiliation': ['Termination For Convenience', 'Post-Termination Services'],
    'responsabilité': ['Cap On Liability', 'Uncapped Liability'],
}
# Les 9 catégories « à plat », et un dictionnaire catégorie -> famille.
NOS_CATEGORIES = [cat for cats in FAMILLES.values() for cat in cats]
FAMILLE_DE = {cat: fam for fam, cats in FAMILLES.items() for cat in cats}

# Afficher les textes un peu longs sans les couper trop tôt dans les tableaux.
pd.set_option('display.max_colwidth', 120)

print('Dossier des données :', CUAD_DIR)
print('Ce dossier existe   :', CUAD_DIR.exists())
print('Version de pandas   :', pd.__version__)

## a. Structure de `CUAD_v1.json`

### a.1 Charger le fichier

**Ce que fait la cellule** : elle lit le fichier JSON (40 Mo) et le transforme en dictionnaire Python.

**Ce qu'on cherche à apprendre** : les clés de premier niveau, la version et le nombre de contrats
(on attend 510).

In [ ]:
chemin_json = CUAD_DIR / 'CUAD_v1.json'
with open(chemin_json, encoding='utf-8') as fichier:
    cuad = json.load(fichier)  # cuad est un dictionnaire Python

print('Clés de premier niveau :', list(cuad.keys()))
print('Version                :', cuad['version'])
print('Nombre de contrats     :', len(cuad['data']))

### a.2 Regarder une entrée de près

**Ce que fait la cellule** : elle descend dans le premier contrat, niveau par niveau :
`data` → `paragraphs` → `context` (le texte) et `qas` (les questions) → `answers`.

**Ce qu'on cherche à apprendre** : la forme exacte d'une question et de ses réponses,
en particulier `answer_start` (la position du passage dans le texte) et `is_impossible`
(vrai quand la catégorie est absente du contrat).

In [ ]:
premier = cuad['data'][0]
print('Titre du contrat       :', premier['title'])
print('Nombre de paragraphes  :', len(premier['paragraphs']))

paragraphe = premier['paragraphs'][0]
print('Longueur du texte      :', len(paragraphe['context']), 'caractères')
print('Nombre de questions    :', len(paragraphe['qas']))
print()
print('Début du texte :')
print(paragraphe['context'][:300])
print()

qa = paragraphe['qas'][0]  # la première question de ce contrat
print('id            :', qa['id'])
print('question      :', qa['question'])
print('is_impossible :', qa['is_impossible'])
print('answers       :', qa['answers'])

### a.3 Mettre les données « à plat » dans des tableaux pandas

**Ce que fait la cellule** : le JSON est imbriqué (des listes dans des listes). Pour compter
facilement, on le parcourt une fois et on construit trois tableaux (*DataFrame*) :

| Tableau | Une ligne par… |
|---|---|
| `contrats` | contrat (titre, longueur du texte) |
| `questions` | couple (contrat, catégorie), avec le nombre de réponses |
| `reponses` | passage annoté (catégorie, position de début et de fin, texte) |

La catégorie est lue dans l'`id`, qui a la forme `<titre>__<catégorie>`.

**Ce qu'on cherche à apprendre** : les grands nombres du jeu de données, et vérifier
deux hypothèses : un seul paragraphe par contrat, et `is_impossible` vrai exactement
quand il n'y a aucune réponse.

In [ ]:
lignes_contrats = []   # une ligne par contrat
lignes_questions = []  # une ligne par couple (contrat, catégorie)
lignes_reponses = []   # une ligne par passage annoté
textes = {}            # titre du contrat -> texte complet (context)

for contrat in cuad['data']:
    titre = contrat['title']
    for paragraphe in contrat['paragraphs']:
        texte = paragraphe['context']
        textes[titre] = texte
        lignes_contrats.append({'titre': titre, 'nb_caracteres': len(texte)})
        for qa in paragraphe['qas']:
            # L'id vaut '<titre>__<catégorie>' : on garde ce qui suit le dernier '__'.
            categorie = qa['id'].rsplit('__', 1)[1]
            lignes_questions.append({
                'titre': titre,
                'categorie': categorie,
                'nb_reponses': len(qa['answers']),
                'is_impossible': qa['is_impossible'],
            })
            for reponse in qa['answers']:
                debut = reponse['answer_start']
                lignes_reponses.append({
                    'titre': titre,
                    'categorie': categorie,
                    'debut': debut,
                    # fin = position juste après le dernier caractère du passage
                    'fin': debut + len(reponse['text']),
                    'texte': reponse['text'],
                })

contrats = pd.DataFrame(lignes_contrats)
questions = pd.DataFrame(lignes_questions)
reponses = pd.DataFrame(lignes_reponses)

print('Paragraphes par contrat (maximum) :', max(len(c['paragraphs']) for c in cuad['data']))
print('Contrats                          :', len(contrats))
print('Titres tous différents ?          :', contrats['titre'].is_unique)
print('Questions (contrat × catégorie)   :', len(questions))
print('Catégories distinctes             :', questions['categorie'].nunique())
print('Questions avec au moins 1 réponse :', (questions['nb_reponses'] > 0).sum())
print('Passages-réponses au total        :', len(reponses))
print()
sans_reponse = questions['nb_reponses'] == 0
print('is_impossible vrai exactement quand il n y a aucune réponse ? :',
      (questions['is_impossible'] == sans_reponse).all())
print('Nos 9 catégories existent bien dans CUAD ? :',
      set(NOS_CATEGORIES) <= set(questions['categorie']))

### a.4 Les positions `answer_start` sont-elles exactes ?

**Ce que fait la cellule** : pour chaque passage, elle découpe le texte du contrat entre
`debut` et `fin`, et compare le résultat au texte annoté.

**Ce qu'on cherche à apprendre** : si les positions sont fiables. C'est essentiel pour notre
règle de traçabilité : chaque sortie devra pointer vers une position exacte dans le texte.
On attend `True` pour tous les passages.

In [ ]:
retrouve = []
for titre, debut, fin, texte in zip(reponses['titre'], reponses['debut'],
                                    reponses['fin'], reponses['texte']):
    retrouve.append(textes[titre][debut:fin] == texte)
reponses['position_ok'] = retrouve

print(reponses['position_ok'].value_counts())
reponses.head()

## b. Longueur des contrats

### b.1 Minimum, médiane, maximum

**Ce que fait la cellule** : elle calcule la longueur minimale, médiane et maximale des contrats,
en caractères, et donne le titre du plus court et du plus long.

**Ce qu'on cherche à apprendre** : l'ordre de grandeur des textes. C'est important pour la suite,
car les modèles Transformer ne lisent qu'un nombre limité de mots à la fois : un contrat entier
devra être découpé en segments (voir D-001).

*La médiane est la valeur du milieu : la moitié des contrats est plus courte, l'autre moitié plus
longue. Contrairement à la moyenne, elle n'est pas tirée vers le haut par quelques contrats très longs.*

In [ ]:
longueurs = contrats['nb_caracteres']
print('Minimum :', longueurs.min(), 'caractères')
print('Médiane :', longueurs.median(), 'caractères')
print('Maximum :', longueurs.max(), 'caractères')
print()
print('Contrat le plus court :', contrats.loc[longueurs.idxmin(), 'titre'])
print('Contrat le plus long  :', contrats.loc[longueurs.idxmax(), 'titre'])

### b.2 Histogramme

**Ce que fait la cellule** : elle range les contrats par tranches de longueur (50 tranches) et
dessine le nombre de contrats dans chaque tranche.

**Ce qu'on cherche à apprendre** : la forme de la répartition. Les contrats sont-ils de longueur
comparable, ou y a-t-il quelques très longs contrats isolés ?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(longueurs, bins=50)
ax.axvline(longueurs.median(), color='black', linestyle='--', label='médiane')
ax.set_xlabel('Longueur du contrat (caractères)')
ax.set_ylabel('Nombre de contrats')
ax.set_title('Longueur des 510 contrats de CUAD')
ax.legend()
plt.tight_layout()
plt.show()

## c. Présence des 41 catégories

### c.1 Tableau

**Ce que fait la cellule** : pour chaque catégorie, elle compte le nombre de contrats **différents**
qui ont au moins un passage annoté, et le pourcentage sur 510. La colonne `notre_famille`
repère nos 9 catégories.

**Ce qu'on cherche à apprendre** : quelles catégories sont fréquentes ou rares. Une catégorie
rare donnera peu d'exemples pour entraîner un modèle.

In [ ]:
# On garde les questions qui ont au moins une réponse,
# puis on compte, pour chaque catégorie, le nombre de contrats différents.
avec_annotation = questions[questions['nb_reponses'] > 0]
contrats_par_categorie = (
    avec_annotation.groupby('categorie')['titre'].nunique()
    # reindex : garder aussi les catégories qui n'auraient aucun contrat (valeur 0)
    .reindex(sorted(questions['categorie'].unique()), fill_value=0)
    .sort_values(ascending=False)
)

tableau_c = pd.DataFrame({
    'nb_contrats': contrats_par_categorie,
    'pourcentage': (contrats_par_categorie / len(contrats) * 100).round(1),
    'notre_famille': contrats_par_categorie.index.map(lambda cat: FAMILLE_DE.get(cat, '')),
})
tableau_c

### c.2 Graphique en barres trié

**Ce que fait la cellule** : elle dessine le même tableau en barres horizontales, de la catégorie
la plus fréquente (en haut) à la plus rare (en bas). Nos catégories sont en orange. La ligne
pointillée marque 510, c'est-à-dire tous les contrats.

**Ce qu'on cherche à apprendre** : situer d'un coup d'œil nos catégories parmi les 41.

In [ ]:
donnees = contrats_par_categorie.sort_values()  # tri croissant : la plus fréquente arrive en haut
couleurs = ['tab:orange' if cat in FAMILLE_DE else 'tab:gray' for cat in donnees.index]

fig, ax = plt.subplots(figsize=(9, 11))
ax.barh(donnees.index, donnees.values, color=couleurs)
ax.axvline(len(contrats), color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Nombre de contrats avec au moins une annotation (sur 510)')
ax.set_title('CUAD : présence de chaque catégorie (orange = nos familles)')
plt.tight_layout()
plt.show()

## d. Exemples réels de passages annotés (nos familles)

**Ce que fait la cellule** : pour chacune de nos 9 catégories, elle tire au hasard 3 passages
annotés et les affiche (300 premiers caractères), avec le contrat d'origine et les positions.
`random_state=0` fixe le tirage : on obtient toujours les mêmes exemples, ce qui rend le résultat
reproductible.

**Ce qu'on cherche à apprendre** : à quoi ressemble concrètement chaque catégorie. Un passage
est-il un mot, une date, une phrase, un paragraphe ? Le style est-il régulier d'un contrat à l'autre ?

*Rappel : une annotation dit seulement « ce passage relève de cette catégorie », pas que la clause
est risquée.*

In [ ]:
for famille, categories in FAMILLES.items():
    print('=' * 90)
    print('FAMILLE :', famille.upper())
    for categorie in categories:
        print('-' * 90)
        print('Catégorie :', categorie)
        passages = reponses[reponses['categorie'] == categorie]
        exemples = passages.sample(n=3, random_state=0)
        for _, ligne in exemples.iterrows():
            print('  •', ligne['titre'][:60], '| positions', ligne['debut'], '→', ligne['fin'])
            # split() puis join : on remplace les sauts de ligne et espaces multiples par un espace
            print('   ', ' '.join(ligne['texte'][:300].split()))

## e. Longueur des passages annotés (nos familles)

**Ce que fait la cellule** : pour nos 9 catégories, elle calcule le nombre de passages et leur
longueur en caractères (médiane, minimum, maximum), triés par médiane.

**Ce qu'on cherche à apprendre** : la taille de ce qu'un modèle devra repérer. Des passages
courts (dates) et longs (clauses entières) n'appellent pas forcément la même méthode, ni la même
taille de segment.

In [ ]:
nos_reponses = reponses[reponses['categorie'].isin(NOS_CATEGORIES)].copy()
nos_reponses['longueur'] = nos_reponses['texte'].str.len()
nos_reponses['famille'] = nos_reponses['categorie'].map(FAMILLE_DE)

tableau_e = (
    nos_reponses.groupby(['famille', 'categorie'])['longueur']
    .agg(['count', 'median', 'min', 'max'])
    .rename(columns={'count': 'nb_passages', 'median': 'mediane'})
    .sort_values('mediane')
)
tableau_e

## f. Contrats avec *Cap On Liability* **et** *Uncapped Liability*

### f.1 Combien ?

**Ce que fait la cellule** : elle construit l'ensemble des contrats annotés pour chaque catégorie,
puis leur **intersection** (les contrats présents dans les deux ensembles).

**Ce qu'on cherche à apprendre** : combien de contrats pourraient déclencher une alerte
« à vérifier » de F3 (comparaison de clauses). Un contrat qui plafonne la responsabilité et
prévoit aussi une responsabilité non plafonnée n'est **pas forcément incohérent** : c'est souvent
un plafond général avec des exceptions (par exemple pour la violation de la confidentialité).
D'où la règle de prudence de F3.

In [ ]:
def contrats_avec(categorie: str) -> set[str]:
    """Renvoie les titres des contrats annotés pour une catégorie.

    Args:
        categorie: Nom exact de la catégorie CUAD, par exemple 'Cap On Liability'.

    Returns:
        L'ensemble des titres des contrats qui ont au moins un passage annoté.
    """
    selection = questions[(questions['categorie'] == categorie) & (questions['nb_reponses'] > 0)]
    return set(selection['titre'])


plafonnee = contrats_avec('Cap On Liability')
non_plafonnee = contrats_avec('Uncapped Liability')
les_deux = plafonnee & non_plafonnee  # & sur deux ensembles = intersection

print('Contrats avec Cap On Liability   :', len(plafonnee))
print('Contrats avec Uncapped Liability :', len(non_plafonnee))
print('Contrats avec les deux           :', len(les_deux))
print('Soit', round(len(les_deux) / len(plafonnee) * 100, 1),
      '% des contrats qui ont un plafond')

### f.2 Un exemple

**Ce que fait la cellule** : elle prend le premier de ces contrats (par ordre alphabétique, pour un
résultat reproductible) et affiche ses passages des deux catégories, avec leurs positions.

**Ce qu'on cherche à apprendre** : en lisant les deux passages, s'agit-il d'une vraie contradiction,
ou d'un plafond général avec des exceptions ?

In [ ]:
exemple = sorted(les_deux)[0]
print('Contrat :', exemple)
for categorie in ['Cap On Liability', 'Uncapped Liability']:
    print()
    print('---', categorie, '---')
    passages = reponses[(reponses['titre'] == exemple) & (reponses['categorie'] == categorie)]
    for _, ligne in passages.iterrows():
        print('  positions', ligne['debut'], '→', ligne['fin'])
        print('   ', ' '.join(ligne['texte'][:500].split()))

### f.3 Les deux catégories annotent-elles le même passage ?

*Cellule ajoutée après la première exécution.* En f.1, les 111 contrats *Uncapped Liability*
ont **tous** aussi *Cap On Liability*. En f.2, le passage affiché pour les deux catégories est
**le même** (mêmes positions).

**Ce que fait la cellule** : pour chaque passage *Uncapped Liability*, elle cherche si le même
contrat a un passage *Cap On Liability* aux **mêmes positions** (début et fin identiques).

**Ce qu'on cherche à apprendre** : si CUAD annote souvent une seule clause dans les deux
catégories (une clause qui plafonne **et** prévoit des exceptions non plafonnées). Si c'est le cas,
la simple présence des deux catégories ne signale pas deux clauses différentes à comparer.

In [ ]:
cap = reponses[reponses['categorie'] == 'Cap On Liability']
uncapped = reponses[reponses['categorie'] == 'Uncapped Liability']

# Un passage est repéré par (contrat, début, fin) : on compare ces triplets entre les deux catégories.
positions_cap = set(zip(cap['titre'], cap['debut'], cap['fin']))
uncapped_aussi_cap = [
    (titre, debut, fin) in positions_cap
    for titre, debut, fin in zip(uncapped['titre'], uncapped['debut'], uncapped['fin'])
]
print('Passages Uncapped Liability                     :', len(uncapped))
print('... dont aussi annotés Cap On Liability         :', sum(uncapped_aussi_cap))

# Par contrat : est-ce que TOUS ses passages Uncapped sont aussi des passages Cap ?
par_contrat = pd.Series(uncapped_aussi_cap, index=uncapped['titre']).groupby(level=0).all()
print('Contrats dont tous les passages Uncapped sont aussi Cap :', int(par_contrat.sum()),
      'sur', len(par_contrat))

## g. Réconcilier 13 823 passages et 13 101 annotations

L'article annonce **13 101** clauses annotées ; le JSON contient **13 823** passages-réponses.

**Hypothèses** (à confirmer ou à rejeter avec les chiffres ci-dessous) :

1. **Doublons** : le même passage (même début, même texte) apparaîtrait plusieurs fois pour une
   même question.
2. **Découpage** : une annotation de l'article (une clause) peut correspondre à plusieurs passages
   dans le JSON. Le README le dit pour *Parties* (4 à 10 morceaux par contrat) et pour les
   annotations non contiguës (`<omitted>`).
3. **Unité de comptage différente** : l'article compterait autre chose, par exemple les couples
   (contrat, catégorie) annotés, ou les éléments du fichier `master_clauses.csv` d'où le JSON
   est dérivé.

### g.1 Plusieurs façons de compter dans le JSON

**Ce que fait la cellule** : elle calcule trois comptages et leur écart avec 13 101, puis la
répartition du nombre de passages par question.

**Ce qu'on cherche à apprendre** : si l'un des comptages tombe sur 13 101 (ou s'en approche),
et donc quelle hypothèse tient.

In [ ]:
nb_passages = len(reponses)
nb_passages_uniques = len(reponses.drop_duplicates(subset=['titre', 'categorie', 'debut', 'texte']))
nb_couples = int((questions['nb_reponses'] > 0).sum())

comparaison = pd.DataFrame({
    'comptage': ['Passages-réponses (JSON)',
                 'Passages sans doublons (JSON)',
                 'Couples (contrat, catégorie) annotés (JSON)'],
    'valeur': [nb_passages, nb_passages_uniques, nb_couples],
})
comparaison['ecart_avec_13101'] = comparaison['valeur'] - 13101
print(comparaison)
print()

# Combien de questions ont 1 passage, 2 passages, 3 passages... ?
print('Nombre de passages par question annotée :')
print(questions.loc[questions['nb_reponses'] > 0, 'nb_reponses'].value_counts().sort_index())

### g.2 Comparer avec `master_clauses.csv`

**Ce que fait la cellule** : le README indique que le JSON est dérivé de `master_clauses.csv`.
Dans ce CSV, chaque cellule de catégorie contient une **liste Python écrite sous forme de texte**,
par exemple `['passage 1', 'passage 2']`. `ast.literal_eval` transforme ce texte en vraie liste,
dont on compte les éléments. La cellule compare ensuite JSON et CSV catégorie par catégorie.

**Ce qu'on cherche à apprendre** : si le CSV contient le même nombre de passages que le JSON,
et dans quelles catégories se concentrent les écarts.

In [ ]:
clauses = pd.read_csv(CUAD_DIR / 'master_clauses.csv')
print('Dimensions du CSV (lignes, colonnes) :', clauses.shape)

# Les colonnes de passages portent le nom exact de la catégorie ;
# les colonnes « -Answer » contiennent des réponses reformulées : on les ignore ici.
colonnes_categories = [col for col in clauses.columns if col in set(questions['categorie'])]
print('Colonnes de catégories trouvées :', len(colonnes_categories), 'sur 41')


def nb_elements(cellule: object) -> int:
    """Compte les passages listés dans une cellule du CSV.

    Args:
        cellule: Contenu brut de la cellule : une liste écrite sous forme de texte, ou une valeur vide.

    Returns:
        Le nombre d'éléments de la liste, ou 0 si la cellule est vide.
    """
    if pd.isna(cellule):
        return 0
    try:
        return len(ast.literal_eval(cellule))
    except (ValueError, SyntaxError):
        return 1  # cellule illisible comme liste : on la compte comme un seul passage


par_categorie = pd.DataFrame({
    'json': reponses.groupby('categorie').size(),
    'csv': pd.Series({col: int(clauses[col].map(nb_elements).sum()) for col in colonnes_categories}),
})
par_categorie['difference'] = par_categorie['json'] - par_categorie['csv']

print('Total JSON :', par_categorie['json'].sum())
print('Total CSV  :', par_categorie['csv'].sum(), '| écart avec 13 101 :', par_categorie['csv'].sum() - 13101)
print()
print('Catégories où JSON et CSV diffèrent le plus :')
par_categorie.sort_values('difference', ascending=False).head(10)

### g.3 Les morceaux `<omitted>` expliquent-ils l'écart ?

*Cellule ajoutée après la première exécution.* En g.2, le CSV contient **exactement 13 101**
passages : c'est donc lui que compte l'article. Reste à comprendre les 722 passages de plus du JSON.

**Hypothèse** : dans le CSV, une annotation non contiguë est écrite en **un seul élément**, avec
`<omitted>` à la place des parties sautées. Le JSON, lui, a besoin de passages exacts du texte :
il découperait cet élément en plusieurs passages, un par morceau. Alors, pour chaque catégorie :

`passages JSON = éléments CSV + nombre de <omitted>`

**Ce que fait la cellule** : elle compte les `<omitted>` dans chaque colonne du CSV et vérifie
cette égalité, catégorie par catégorie et au total.

**Ce qu'on cherche à apprendre** : si l'hypothèse 2 (découpage) explique tout l'écart.

In [ ]:
def nb_omitted(cellule: object) -> int:
    """Compte les marqueurs <omitted> dans une cellule du CSV.

    Args:
        cellule: Contenu brut de la cellule, ou une valeur vide.

    Returns:
        Le nombre d'occurrences de '<omitted>' (0 si la cellule est vide).
    """
    if pd.isna(cellule):
        return 0
    return str(cellule).count('<omitted>')


par_categorie['omitted'] = pd.Series(
    {col: int(clauses[col].map(nb_omitted).sum()) for col in colonnes_categories}
)
par_categorie['csv_plus_omitted'] = par_categorie['csv'] + par_categorie['omitted']
par_categorie['reste'] = par_categorie['json'] - par_categorie['csv_plus_omitted']

print('Total <omitted> dans le CSV        :', par_categorie['omitted'].sum())
print('Éléments CSV + <omitted>           :', par_categorie['csv_plus_omitted'].sum())
print('Passages JSON                      :', par_categorie['json'].sum())
print('Catégories où l égalité est exacte :', int((par_categorie['reste'] == 0).sum()), 'sur 41')
print()
print('Catégories où il reste un écart :')
par_categorie[par_categorie['reste'] != 0].sort_values('reste')

## h. Le champ `context` est-il identique au fichier TXT ?

### h.1 Sur 10 contrats

**Ce que fait la cellule** : pour 10 contrats tirés au hasard (tirage fixé), elle ouvre le fichier
`full_contract_txt/<titre>.txt` et le compare au `context` du JSON de deux façons :

- **identique** : caractère pour caractère ;
- **identique aux espaces près** : après avoir remplacé toute suite d'espaces et de sauts de ligne
  par un seul espace.

**Ce qu'on cherche à apprendre** : si les positions `answer_start` (calculées sur `context`)
valent aussi pour les fichiers TXT. Si les textes diffèrent, il faudra toujours utiliser `context`
comme référence, ou aligner les positions (limite 3 de `data_provenance.md`).

In [ ]:
dossier_txt = CUAD_DIR / 'full_contract_txt'


def comparer_au_txt(titre: str) -> dict[str, object]:
    """Compare le texte du JSON (context) d'un contrat avec son fichier TXT.

    Args:
        titre: Titre du contrat dans CUAD_v1.json.

    Returns:
        Un dictionnaire : fichier trouvé, identique, identique aux espaces près, longueurs.
    """
    chemin = dossier_txt / (titre + '.txt')
    if not chemin.exists():
        return {'titre': titre, 'fichier_trouve': False}
    texte_txt = chemin.read_text(encoding='utf-8', errors='replace')
    texte_json = textes[titre]
    return {
        'titre': titre,
        'fichier_trouve': True,
        'identique': texte_txt == texte_json,
        'identique_aux_espaces_pres': ' '.join(texte_txt.split()) == ' '.join(texte_json.split()),
        'longueur_txt': len(texte_txt),
        'longueur_json': len(texte_json),
    }


echantillon = contrats['titre'].sample(n=10, random_state=0)
pd.DataFrame([comparer_au_txt(titre) for titre in echantillon])

### h.2 Sur les 510 contrats

**Ce que fait la cellule** : la même comparaison pour tous les contrats, résumée en trois chiffres.

**Ce qu'on cherche à apprendre** : si le résultat de l'échantillon se généralise, et si chaque
titre du JSON a bien son fichier TXT.

In [ ]:
comparaison_txt = pd.DataFrame([comparer_au_txt(titre) for titre in contrats['titre']])
trouves = comparaison_txt[comparaison_txt['fichier_trouve']]

print('Fichiers TXT trouvés                :', len(trouves), 'sur', len(comparaison_txt))
print('Identiques caractère pour caractère :', int(trouves['identique'].astype(bool).sum()))
print('Identiques aux espaces près         :', int(trouves['identique_aux_espaces_pres'].astype(bool).sum()))

### h.3 Où se trouve la première différence ?

**Ce que fait la cellule** : s'il existe des contrats non identiques, elle prend le premier et
affiche les 40 caractères avant et après la première différence. `repr()` rend visibles les
caractères invisibles (espaces, sauts de ligne, caractères spéciaux).

**Ce qu'on cherche à apprendre** : la **nature** des différences (espaces ? sauts de ligne ?
caractère d'encodage en début de fichier ?), pour savoir si elles sont gênantes.

In [ ]:
def premiere_difference(a: str, b: str) -> int:
    """Renvoie la position du premier caractère qui diffère entre deux textes.

    Args:
        a: Premier texte.
        b: Second texte.

    Returns:
        La position de la première différence, ou -1 si les textes sont identiques.
    """
    for position, (car_a, car_b) in enumerate(zip(a, b)):
        if car_a != car_b:
            return position
    return -1 if len(a) == len(b) else min(len(a), len(b))


differents = trouves[~trouves['identique'].astype(bool)]
if len(differents) == 0:
    print('Aucune différence : context et TXT sont identiques pour tous les contrats trouvés.')
else:
    titre = differents['titre'].iloc[0]
    texte_txt = (dossier_txt / (titre + '.txt')).read_text(encoding='utf-8', errors='replace')
    texte_json = textes[titre]
    pos = premiere_difference(texte_txt, texte_json)
    print('Contrat :', titre)
    print('Première différence à la position', pos)
    print('TXT  :', repr(texte_txt[max(pos - 40, 0):pos + 40]))
    print('JSON :', repr(texte_json[max(pos - 40, 0):pos + 40]))

## Conclusions

*Rédigées après l'exécution complète du 2026-11-22 (noyau `.venv`, Python 3.11.18, pandas 3.1.1).*
*Aucun extrait de contrat n'est recopié ici (dépôt public) : on décrit, on ne cite pas.*

- **a. Structure** : 510 contrats, 41 catégories, donc 20 910 questions (contrat × catégorie),
  dont 6 702 ont au moins une réponse, pour **13 823 passages**. Un seul paragraphe par contrat :
  **oui**, le `context` est le texte entier. `is_impossible` est vrai exactement quand il n'y a
  aucune réponse. Positions `answer_start` exactes : **13 823 sur 13 823**.
- **b. Longueur des contrats** : de 645 à 338 211 caractères, médiane 33 143. La répartition est
  très asymétrique : la plupart des contrats sont courts, quelques-uns très longs.
  **Conséquence** : un contrat médian dépasse largement ce qu'un Transformer lit d'un coup
  (512 *tokens*, soit environ 2 000 caractères). Il faut découper en segments (D-001).
- **c. Présence des catégories** : les plus fréquentes sont *Document Name* (510), *Parties* (509)
  et *Agreement Date* (470) ; les plus rares *Source Code Escrow* (13) et *Price Restrictions* (15).
  Nos 9 catégories vont de 470 contrats (*Agreement Date*, 92 %) à 111 (*Uncapped Liability* et
  *Notice Period To Terminate Renewal*, 22 %). **Conséquence** : dans la plupart des contrats,
  la majorité des catégories est absente. Le modèle devra souvent répondre « rien » : la classe
  « absente » domine, à prendre en compte dans l'entraînement et les métriques.
- **d. Exemples** : la famille **durée** mêle des dates très courtes (parfois un gabarit vide entre
  crochets au lieu d'une date) et des phrases complètes (durée initiale, renouvellement automatique,
  préavis de non-renouvellement). **Résiliation** et **responsabilité** sont des phrases ou
  paragraphes entiers. *Cap On Liability* est une notion large : on y trouve aussi des exclusions
  de dommages indirects et des remèdes exclusifs, pas seulement des plafonds chiffrés.
- **e. Longueur des passages** : médiane de 16 caractères (*Agreement Date*) à 423
  (*Uncapped Liability*) ; maximum 2 783 (*Renewal Term*). **Conséquence** : deux types de tâche,
  repérer une valeur courte (date) ou une clause entière. Une même méthode ou une même taille de
  segment ne conviendra peut-être pas aux deux.
- **f. Cap + Uncapped** : 111 contrats sur 275 avec un plafond (40,4 %), et surtout **les 111**
  contrats *Uncapped* ont aussi *Cap*. En f.3 : **125 des 167** passages *Uncapped* sont
  **exactement** des passages *Cap* (mêmes positions). Dans 73 contrats sur 111, tous les passages
  *Uncapped* sont aussi *Cap*. Dans l'exemple lu, c'est **une seule clause** : une exclusion des
  dommages indirects **avec une exception** pour la violation d'une section précise. Ce n'est pas
  une contradiction, c'est une limite avec exception. **Conséquence pour F3** : la présence des
  deux catégories ne doit **pas** déclencher d'alerte ; F3 doit comparer des **clauses distinctes**
  et rester prudent.
- **g. 13 823 et 13 101** : le CSV `master_clauses.csv` contient **exactement 13 101** éléments.
  C'est donc l'unité de l'article (hypothèse 3), et il n'y a **aucun doublon** dans le JSON
  (hypothèse 1 rejetée). L'écart de 722 vient du **découpage** (hypothèse 2) : une annotation non
  contiguë est un seul élément dans le CSV (avec `<omitted>`) mais plusieurs passages dans le JSON.
  Éléments CSV + `<omitted>` = 13 834, à **11 près** des 13 823 passages ; égalité exacte pour 32
  catégories sur 41. Écart restant : 11 passages, dans 9 catégories (sans doute des `<omitted>` en
  bord de texte ou des morceaux fusionnés ; non vérifié, sans incidence pour nous).
- **h. `context` et TXT** : identiques **caractère pour caractère pour 510 contrats sur 510**.
  **Conséquence** : les positions CUAD valent aussi pour `full_contract_txt/`. Mais notre extraction
  PyMuPDF (F1) produira un autre texte : l'alignement reste nécessaire (limite 3).

**Rappel** : ces constats sont descriptifs. Ils n'ont servi à régler aucun modèle. Le découpage
entraînement / validation / test reste à décider (D-004).

**Points à reporter** :
- `docs/data_provenance.md`, limite 3 : `context` = TXT (510/510), positions exactes (13 823/13 823).
- `docs/data_provenance.md`, limite 9 : 13 101 = éléments du CSV ; écart expliqué par `<omitted>`
  à 11 passages près.
- `docs/data_provenance.md`, limite 8 : le découpage officiel existe dans `data.zip` du dépôt
  GitHub (`train_separate_questions.json`, `test.json`, d'après `run.sh`) ; pas de jeu de validation.
- `docs/decisions.md` : D-004 (découpage), et une note pour F3 (Cap et Uncapped se recouvrent).